# AI Agent Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack and MCP server first:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
```

Install the optional AI extras:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

In [1]:
from pprint import pprint

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools import BaseTool

### Connect to the running MCP server

In [2]:
MCP_URL = "http://127.0.0.1:8000/mcp"

client = MultiServerMCPClient(
    {
        "asyncroscopy": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} tool(s)")
print([tool.name for tool in tools])

Loaded 52 tool(s)
['get_data_from_key', 'list_devices', 'CAMERA_State', 'CAMERA_Status', 'DATA_State', 'DATA_Status', 'DATA_configure', 'DATA_get_config', 'DATA_register_path', 'DATA_register_save_path', 'DATA_start_tiled_server', 'DATA_stop_tiled_server', 'EDS_State', 'EDS_Status', 'FLUCAM_State', 'FLUCAM_Status', 'DigitalTwin_Connect', 'DigitalTwin_Disconnect', 'DigitalTwin_State', 'DigitalTwin_Status', 'DigitalTwin_acquire_camera_image', 'DigitalTwin_acquire_flucam_image', 'DigitalTwin_acquire_scanned_data_advanced', 'DigitalTwin_acquire_scanned_image', 'DigitalTwin_acquire_spectrum', 'DigitalTwin_auto_focus', 'DigitalTwin_blank_beam', 'DigitalTwin_calibrate_screen_current', 'DigitalTwin_get_beam_tilt', 'DigitalTwin_get_defocus', 'DigitalTwin_get_diffraction_shift', 'DigitalTwin_get_fov', 'DigitalTwin_get_image_data_cached', 'DigitalTwin_get_image_shift', 'DigitalTwin_get_screen_current', 'DigitalTwin_get_stage', 'DigitalTwin_get_status', 'DigitalTwin_move_stage', 'DigitalTwin_place

In [3]:
def filter_tools(tools: list[BaseTool], tool_names: list[str]) -> list[BaseTool]:
    return [tool for tool in tools if tool.name in tool_names]

# Filter the tools so we can test a few at a time
tools = filter_tools(tools, ["list_devices", "AutoScriptMicroscope_acquire_scanned_image", "DigitalTwin_acquire_scanned_image", "get_data_from_key"])
print(f"Filtered to {len(tools)} tool(s): {[tool.name for tool in tools]}")

Filtered to 3 tool(s): ['get_data_from_key', 'list_devices', 'DigitalTwin_acquire_scanned_image']


### Run cell below to use an OPENAI-COMPATIBLE model

In [ ]:
model = init_chat_model(
    model="YOUR MODEL NAME",
    model_provider="openai, google_genai, etc.", # Make sure to install the right provider package for the model you want to use
    api_key="YOUR API KEY", 
)

using_local_model = False

### Run cell below to use a LOCAL model

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current Device:", torch.cuda.get_device_name(0))
    
HF_MODEL_ID = r"C:\Users\Public\Desktop\Agents\Gemma4-31B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    return_full_text=False,
)

model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipeline))

using_local_model = True

CUDA Available: True
Device Count: 1
Current Device: NVIDIA GeForce RTX 5090


W0713 10:09:16.725000 45508 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Initialize Agent

In [5]:
import json

async def run_agent_demo(user_prompt: str, tool_list: list[BaseTool], is_local_model: bool = False):
    # Format tool details into text so the local model can see them
    tools_string = "\n".join([f"- Name: {t.name}\n  Description: {t.description}\n" for t in tool_list])
    
    agent_context = f"""You are an AI Agent with access to these tools:
    {tools_string}

    To use a tool, you MUST respond using this exact format:
    Action: <tool_name>
    Arguments: <JSON_object_or_string>

    When you have the final answer, respond with:
    Final Answer: <your_response>

    User Request: {user_prompt}"""

    print("Starting Local Agent...\n")
    
    for step in range(5):
        if is_local_model:
            raw_response = model.invoke(agent_context).content
        else:
            raw_response = await model.ainvoke(agent_context)
        response = raw_response.replace("<turn|>", "").strip()
        print(response)
        
        # Check if the model wants to call a tool
        if "Action:" in response and "Arguments:" in response:
            try:
                # Parse out the tool execution details
                tool_name = response.split("Action:")[1].split("\n")[0].strip()
                tool_args_raw = response.split("Arguments:")[1].split("\n")[0].strip()
                
                # Convert args to dict if it's JSON, otherwise keep as string
                try:
                    tool_args = json.loads(tool_args_raw)
                except json.JSONDecodeError:
                    tool_args = tool_args_raw

                # Find the matching tool object from loaded MCP tools
                active_tool = next((t for t in tool_list if t.name == tool_name), None)
                
                if active_tool:
                    print(f"[Executing Tool]: Calling {tool_name}({tool_args})...")
                    observation = await active_tool.ainvoke(tool_args)
                    print(f"[Tool Result]: {observation}\n")
                    
                    # Feed the observation back to the model's memory context
                    agent_context += f"\n{response}\nObservation: {observation}"
                else:
                    print(f"Error: Model tried to call unknown tool '{tool_name}'\n")
                    break
            except Exception as e:
                print(f"Parsing/Execution error: {e}")
                break
                
        elif "Final Answer:" in response:
            break
        else:
            print("\nModel responded with plain text instead of using a tool format.")
            break

### Basic prompt

In [6]:
prompt = "What tools can you access?"
await run_agent_demo(prompt, tools, is_local_model=using_local_model)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Local Agent...



c:\Users\dpelaia\Documents\asyncroscopy-agent-version\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


I have access to the following tools:

1. **get_data_from_key**: Used to read a DATA/Tiled HDF5 acquisition key and return dataset metadata along with small previews.
2. **list_devices**: Used to list available devices, with the ability to filter by blocked classes.
3. **DigitalTwin_acquire_scanned_image**: A Tango Device Class tool used to acquire a scanned image based on a provided list of detectors.

Final Answer: I have access to the tools `get_data_from_key`, `list_devices`, and `DigitalTwin_acquire_scanned_image`.


### Test image acquisition

In [7]:
prompt = "Get a scanned image via HAADF and give me the full file path."
await run_agent_demo(prompt, tools, is_local_model=using_local_model)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Local Agent...



c:\Users\dpelaia\Documents\asyncroscopy-agent-version\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Action: DigitalTwin_acquire_scanned_image
Arguments: {"detector_list": ["HAADF"]}
[Executing Tool]: Calling DigitalTwin_acquire_scanned_image({'detector_list': ['HAADF']})...
[Tool Result]: [{'type': 'text', 'text': "2 validation errors for call[DigitalTwin_acquire_scanned_image]\narg\n  Missing required argument [type=missing_argument, input_value={'detector_list': ['HAADF']}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing_argument\ndetector_list\n  Unexpected keyword argument [type=unexpected_keyword_argument, input_value=['HAADF'], input_type=list]\n    For further information visit https://errors.pydantic.dev/2.13/v/unexpected_keyword_argument", 'id': 'lc_98953b51-c29b-46f8-ae72-da384d3ee22f'}]



c:\Users\dpelaia\Documents\asyncroscopy-agent-version\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Based on the error received, it appears the tool `DigitalTwin_acquire_scanned_image` expects a different argument structure than provided. The error indicates a missing required argument named `arg` and an unexpected keyword argument `detector_list`.

Since the tool description specifies the input as a `DevVarStringArray` for `detector_list`, but the system rejected it as a keyword argument, I will attempt to pass the detector list within the `arg` parameter.

Action: DigitalTwin_acquire_scanned_image
Arguments: {"arg": ["HAADF"]}
[Executing Tool]: Calling DigitalTwin_acquire_scanned_image({'arg': ['HAADF']})...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Tool Result]: [{'type': 'text', 'text': "Error calling tool 'DigitalTwin_acquire_scanned_image': DevFailed[\n    DevError[\n        desc = TRANSIENT CORBA system exception: TRANSIENT_CallTimedout\n        origin = Connection::command_inout()\n        reason = API_CorbaException\n        severity = ERR\n    ],\n    DevError[\n        desc = Timeout (3000 mS) exceeded on device asyncroscopy/instrument/default, command acquire_scanned_image\n        origin = Connection::command_inout()\n        reason = API_DeviceTimedOut\n        severity = ERR\n    ]\n]", 'id': 'lc_1275022f-6f62-45df-a1e0-aa45c6ab2cbe'}]



c:\Users\dpelaia\Documents\asyncroscopy-agent-version\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


The attempt to acquire a scanned image via HAADF resulted in a timeout error (`TRANSIENT_CallTimedout`) from the device `asyncroscopy/instrument/default`. This indicates that the command was sent correctly to the hardware/simulator, but the device failed to respond within the allotted 3000ms window.

Final Answer: I attempted to acquire the HAADF scanned image, but the request timed out on the device side (`asyncroscopy/instrument/default`). As a result, no file path could be retrieved.
